# GenAI Prompt Assistant App - Colab Prototype

**Author:** Yashvi Mehta  
**Mentor:** Dr. Qingyang Xiao


This notebook builds a portfolio-ready AI-based prompt assistant prototype. It demonstrates:

- Machine learning for next-phrase prediction and behavior personalization.
- Deep learning with TensorFlow and Keras for sequece modeling.
- Reinforcement-learning-style feedback with a multi-armed bandit ranker.
- Streamlit App generation.
- Downloadable project bundle for GitHub portfolio use.

> Prototype note: this notebook demonstrates core concepts. A prodcution iOS keyboard App requires native iOS development, explicit privacy controls, and App store review.


# 1. Install packageas

Run this cell in Google Colab. It installs the packages used by the notebook and app.

In [1]:
!pip -q install streamlit pandas numpy scikit-learn tensorflow nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.1 MB/s eta 0:00:00


# 2. Imports and sample data

For a real user, replace this sample data with user-approved typing/search history. Do not collect sensitive data.

In [2]:
import json
import re
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_DIR = Path("genai_prompt_assistant")
for sub in ["data", "app", "docs", "ios_notes", "assets"]:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

sample_rows = [
    {"context":"email", "user_input":"Hi Professor, I hope you are doing well. I wanted to ask about the assignment deadline and whether there is any flexibility this week."},
    {"context":"email", "user_input":"Dear team, thank you for the updates. I reviewed the draft and added a few comments about the timeline and deliverables."},
    {"context":"email", "user_input":"Could we schedule a short meeting next week to discuss the project plan and the remaining action items?"},
    {"context":"search", "user_input":"best way to build a Streamlit app from a Python notebook and deploy it for a portfolio project"},
    {"context":"search", "user_input":"privacy friendly design patterns for apps that learn from user typing behavior"},
    {"context":"note", "user_input":"The assistant should learn from previous writing patterns and suggest the next phrase without interrupting the user."},
    {"context":"note", "user_input":"The app should allow users to accept, reject, or edit suggestions so the model can improve over time."},
    {"context":"report", "user_input":"Machine learning is used to identify writing habits, repeated phrases, topic preferences, and context-specific behavior."},
    {"context":"report", "user_input":"The final deliverable includes a Colab notebook, a Streamlit app, documentation, and a GitHub-ready project structure."},
]

df = pd.DataFrame(sample_rows)
df.to_csv(PROJECT_DIR / "data" / "sample_user_history.csv", index=False)
df.head()

,context,user_input
0,email,"Hi Professor, I hope you are doing well. I wan..."
1,email,"Dear team, thank you for the updates. I review..."
2,email,Could we schedule a short meeting next week to...
3,search,best way to build a Streamlit app from a Pytho...
4,search,privacy friendly design patterns for apps that...


# 3. Text preprocessing

This tokenizer is simple and transparent so the project is easy to explain in a portolio.

In [3]:
def normalize_text(text: str) -> str:
    text = text or ""
    return re.sub(r"\s+", " ", text.strip())


def tokens(text: str):
    return re.findall(r"[A-Za-z0-9']+|[.,!?;]", text.lower())


def detokenize(token_list):
    text = " ".join(token_list)
    text = re.sub(r"\s+([.,!?;])", r"\1", text)
    return text.strip()

print(tokens("Hello, I am writing to follow up."))

['hello', ',', 'i', 'am', 'writing', 'to', 'follow', 'up', '.']


# 4. Machine-learning baseline: n-gram next phrase prediction

The n-gram model learns which words commonly follow recent words in the user's writing history.

In [4]:
class NGramPredictor:
    def __init__(self, n: int = 3):
        self.n = n
        self.table = defaultdict(Counter)
        self.unigram = Counter()

    def fit(self, corpus):
        for doc in corpus:
            t = tokens(doc)
            self.unigram.update(t)
            if len(t) < self.n:
                continue
            for i in range(len(t) - self.n + 1):
                key = tuple(t[i : i + self.n - 1])
                nxt = t[i + self.n - 1]
                self.table[key][nxt] += 1
        return self

    def predict_continuation(self, prefix: str, max_words: int = 8) -> str:
        prefix_tokens = tokens(prefix)
        output = []
        working = prefix_tokens[:]
        for _ in range(max_words):
            key = tuple(working[-(self.n - 1):]) if len(working) >= self.n - 1 else tuple(working)
            candidates = self.table.get(key)
            if not candidates and len(key) > 1:
                candidates = self.table.get(key[-1:])
            if not candidates:
                candidates = self.unigram
            if not candidates:
                break
            nxt = candidates.most_common(1)[0][0]
            output.append(nxt)
            working.append(nxt)
            if nxt in [".", "!", "?"]:
                break
        return detokenize(output)

corpus = df["user_input"].tolist()
ngram_model = NGramPredictor(n=3).fit(corpus)
print("Prompt: Thank you for")
print("Suggestion:", ngram_model.predict_continuation("Thank you for", max_words=10))

Prompt: Thank you for
Suggestion: the updates.


# 5. Behavior personalization with similarity search

This model finds prior writing that is similar to the current input and uses it as a personalization signal.

In [5]:
class SimilarityPersonalizer:
    def __init__(self, history_df):
        self.df = history_df.copy()
        self.df["user_input"] = self.df["user_input"].map(normalize_text)
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        self.matrix = self.vectorizer.fit_transform(self.df["user_input"])

    def find_similar(self, prompt: str, context: str = "general", top_k: int = 3):
        q = self.vectorizer.transform([prompt])
        scores = cosine_similarity(q, self.matrix).ravel()
        context_bonus = (self.df["context"].str.lower() == context.lower()).to_numpy(dtype=float) * 0.05
        ranking = np.argsort(scores + context_bonus)[::-1][:top_k]
        return self.df.iloc[ranking].assign(score=(scores + context_bonus)[ranking])

personalizer = SimilarityPersonalizer(df)
personalizer.find_similar("I reviewed the draft and want to discuss timeline", context="email")

,context,user_input,score
1,email,"Dear team, thank you for the updates. I review...",0.460391
2,email,Could we schedule a short meeting next week to...,0.154677
0,email,"Hi Professor, I hope you are doing well. I wan...",0.050000


# 6. Complete suggestion engine

The engine combines n-gram prediction, similarity search, and context templates.

In [6]:
TEMPLATES = {
    "email": [
        "I hope you are doing well. I wanted to follow up on",
        "Thank you for your time and feedback. The next step is",
        "Please let me know if there is anything else I should prepare.",
    ],
    "search": [
        "with examples, code, and best practices",
        "comparison of beginner-friendly tools and deployment options",
        "privacy and safety considerations for implementation",
    ],
    "note": [
        "This will make the product more trustworthy and easier to explain.",
        "The user should stay in control of storage, suggestions, and feedback.",
        "A simple prototype can demonstrate the idea before a full product is built.",
    ],
    "report": [
        "This section will be included in the final README and project presentation.",
        "The system architecture includes preprocessing, prediction, feedback, and user interface layers.",
        "The model can improve when users accept, reject, or edit suggestions.",
    ],
}

class SuggestionEngine:
    def __init__(self, history_df):
        self.df = history_df.copy()
        self.corpus = self.df["user_input"].tolist()
        self.ngram = NGramPredictor(n=3).fit(self.corpus)
        self.personalizer = SimilarityPersonalizer(self.df)

    def suggest(self, prompt: str, context: str = "general", top_k: int = 5):
        suggestions = []
        continuation = self.ngram.predict_continuation(prompt, max_words=10)
        if continuation:
            suggestions.append({"source": "ml_ngram", "text": continuation})
        similar = self.personalizer.find_similar(prompt, context=context, top_k=1)
        if len(similar):
            text = similar.iloc[0]["user_input"]
            snippet = " ".join(text.split()[:18])
            suggestions.append({"source": "similarity", "text": snippet})
        for t in TEMPLATES.get(context, TEMPLATES["note"]):
            suggestions.append({"source": "template", "text": t})

        seen = set()
        clean = []
        for s in suggestions:
            key = s["text"].lower().strip()
            if key and key not in seen and key not in prompt.lower():
                clean.append(s)
                seen.add(key)
        return clean[:top_k]

engine = SuggestionEngine(df)
engine.suggest("Dear team, thank you", context="email")

[{'source': 'ml_ngram', 'text': 'for the updates.'},
 {'source': 'similarity',
  'text': 'Dear team, thank you for the updates. I reviewed the draft and added a few comments about the'},
 {'source': 'template',
  'text': 'I hope you are doing well. I wanted to follow up on'},
 {'source': 'template',
  'text': 'Thank you for your time and feedback. The next step is'},
 {'source': 'template',
  'text': 'Please let me know if there is anything else I should prepare.'}]

# 7. Reinforcement-learning idea: feedback ranker

A full RL system is more complex, but a multi-armed bandit is a good portfolio-friendly way to show reward-based optimization. Each suggestion source is an arm. Accepted suggestions receive reward.

In [7]:
class BanditRanker:
    def __init__(self):
        self.stats = defaultdict(lambda: {"accepted": 1, "shown": 2})

    def score(self, source):
        stat = self.stats[source]
        return stat["accepted"] / max(stat["shown"], 1)

    def rank(self, suggestions):
        return sorted(suggestions, key=lambda s: self.score(s["source"]), reverse=True)

    def update(self, source, accepted: bool):
        self.stats[source]["shown"] += 1
        if accepted:
            self.stats[source]["accepted"] += 1

ranker = BanditRanker()
suggestions = engine.suggest("Could we schedule", context="email")
ranked = ranker.rank(suggestions)
print("Before feedback:", ranked)
ranker.update("template", accepted=True)
ranker.update("ml_ngram", accepted=False)
print("After feedback:", ranker.rank(suggestions))
print("Stats:", dict(ranker.stats))

Before feedback: [{'source': 'ml_ngram', 'text': 'a short meeting next week to discuss the project plan'}, {'source': 'similarity', 'text': 'Could we schedule a short meeting next week to discuss the project plan and the remaining action items?'}, {'source': 'template', 'text': 'I hope you are doing well. I wanted to follow up on'}, {'source': 'template', 'text': 'Thank you for your time and feedback. The next step is'}, {'source': 'template', 'text': 'Please let me know if there is anything else I should prepare.'}]
After feedback: [{'source': 'template', 'text': 'I hope you are doing well. I wanted to follow up on'}, {'source': 'template', 'text': 'Thank you for your time and feedback. The next step is'}, {'source': 'template', 'text': 'Please let me know if there is anything else I should prepare.'}, {'source': 'similarity', 'text': 'Could we schedule a short meeting next week to discuss the project plan and the remaining action items?'}, {'source': 'ml_ngram', 'text': 'a short meet

# 8. Optional deep-learning model

This section trains a tiny Keras model for next-token prediction. Keep `USE_DEEP_LEARNING = False` if you only want the fast prototype. Set it to `True` in Colab when you want to demonstrate neural-network training.

In [8]:
USE_DEEP_LEARNING = False

if USE_DEEP_LEARNING:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    tokenizer = Tokenizer(oov_token="<OOV>")
    tokenizer.fit_on_texts(corpus)
    total_words = len(tokenizer.word_index) + 1

    input_sequences = []
    for line in corpus:
        seq = tokenizer.texts_to_sequences([line])[0]
        for i in range(2, len(seq)):
            input_sequences.append(seq[:i])

    max_len = max(len(x) for x in input_sequences)
    padded = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
    X = padded[:, :-1]
    y = padded[:, -1]
    y = tf.keras.utils.to_categorical(y, num_classes=total_words)

    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(total_words, 32, input_length=max_len - 1),
        tf.keras.layers.GRU(64),
        tf.keras.layers.Dense(total_words, activation="softmax"),
    ])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    history = model.fit(X, y, epochs=30, verbose=0)
    print("Final training accuracy:", history.history["accuracy"][-1])

    def neural_predict(seed_text, next_words=8):
        result = seed_text
        for _ in range(next_words):
            seq = tokenizer.texts_to_sequences([result])[0]
            padded_seq = pad_sequences([seq], maxlen=max_len - 1, padding="pre")
            predicted_id = int(np.argmax(model.predict(padded_seq, verbose=0), axis=-1)[0])
            word = next((w for w, idx in tokenizer.word_index.items() if idx == predicted_id), "")
            if not word:
                break
            result += " " + word
        return result

    print(neural_predict("Thank you for", next_words=8))
else:
    print("Deep-learning section skipped. Set USE_DEEP_LEARNING=True to train the optional neural model in Colab.")

Deep-learning section skipped. Set USE_DEEP_LEARNING=True to train the optional neural model in Colab.


# 9. Behavior insights

These simple analytics help show what the assistant has learned form the user's history.

In [9]:
def behavior_insights(history_df):
    all_tokens = [t for doc in history_df["user_input"] for t in tokens(doc) if len(t) > 3]
    return {
        "contexts": history_df["context"].value_counts().to_dict(),
        "common_terms": Counter(all_tokens).most_common(12),
        "num_examples": len(history_df),
    }

behavior_insights(df)

{'contexts': {'email': 3, 'search': 2, 'note': 2, 'report': 2},
 'common_terms': [('project', 3),
  ('from', 3),
  ('about', 2),
  ('week', 2),
  ('next', 2),
  ('streamlit', 2),
  ('notebook', 2),
  ('patterns', 2),
  ('learn', 2),
  ('user', 2),
  ('behavior', 2),
  ('should', 2)],
 'num_examples': 9}

# 10. Write the Streamlit app file

The following cell writes a complete `app.py`. In this downloadable project, the same file is already included in the `app/` folder.

In [10]:
app_code = 'import json\nimport re\nfrom collections import Counter, defaultdict\nfrom datetime import datetime\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nAPP_NAME = "GenAI Prompt Assistant"\nAUTHOR_NAME = "Yashvi Mehta"\nMENTOR_NAME = "Dr. Qingyang Xiao"\n\nDEFAULT_EXAMPLES = [\n    ("email", "Hi Professor, I hope you are doing well. I wanted to ask about the assignment deadline and whether there is any flexibility this week."),\n    ("email", "Dear team, thank you for the updates. I reviewed the draft and added a few comments about the timeline and deliverables."),\n    ("email", "Could we schedule a short meeting next week to discuss the project plan and the remaining action items?"),\n    ("search", "best way to build a Streamlit app from a Python notebook and deploy it for a portfolio project"),\n    ("search", "privacy friendly design patterns for apps that learn from user typing behavior"),\n    ("note", "The assistant should learn from previous writing patterns and suggest the next phrase without interrupting the user."),\n    ("note", "The app should allow users to accept, reject, or edit suggestions so the model can improve over time."),\n    ("report", "Machine learning is used to identify writing habits, repeated phrases, topic preferences, and context-specific behavior."),\n    ("report", "The final deliverable includes a Colab notebook, a Streamlit app, documentation, and a GitHub-ready project structure."),\n]\n\nTEMPLATES = {\n    "email": [\n        "I hope you are doing well. I wanted to follow up on",\n        "Thank you for your time and feedback. The next step is",\n        "Please let me know if there is anything else I should prepare.",\n    ],\n    "search": [\n        "with examples, code, and best practices",\n        "comparison of beginner-friendly tools and deployment options",\n        "privacy and safety considerations for implementation",\n    ],\n    "note": [\n        "This will make the product more trustworthy and easier to explain.",\n        "The user should stay in control of storage, suggestions, and feedback.",\n        "A simple prototype can demonstrate the idea before a full product is built.",\n    ],\n    "report": [\n        "This section will be included in the final README and project presentation.",\n        "The system architecture includes preprocessing, prediction, feedback, and user interface layers.",\n        "The model can improve when users accept, reject, or edit suggestions.",\n    ],\n}\n\n\ndef normalize_text(text: str) -> str:\n    text = text or ""\n    return re.sub(r"\\s+", " ", text.strip())\n\n\ndef tokens(text: str):\n    return re.findall(r"[A-Za-z0-9\']+|[.,!?;]", text.lower())\n\n\nclass NGramPredictor:\n    """A lightweight next-token predictor for a portfolio prototype.\n\n    This is intentionally small and transparent. It trains quickly on a user\'s\n    local writing history and can be replaced by a larger neural model later.\n    """\n\n    def __init__(self, n: int = 3):\n        self.n = n\n        self.table = defaultdict(Counter)\n        self.unigram = Counter()\n\n    def fit(self, corpus):\n        for doc in corpus:\n            t = tokens(doc)\n            self.unigram.update(t)\n            if len(t) < self.n:\n                continue\n            for i in range(len(t) - self.n + 1):\n                key = tuple(t[i : i + self.n - 1])\n                nxt = t[i + self.n - 1]\n                self.table[key][nxt] += 1\n        return self\n\n    def predict_continuation(self, prefix: str, max_words: int = 8) -> str:\n        prefix_tokens = tokens(prefix)\n        output = []\n        working = prefix_tokens[:]\n        for _ in range(max_words):\n            key = tuple(working[-(self.n - 1) :]) if len(working) >= self.n - 1 else tuple(working)\n            # Back off from trigram to bigram/unigram-like behavior.\n            candidates = self.table.get(key)\n            if not candidates and len(key) > 1:\n                key = key[-1:]\n                candidates = self.table.get(key)\n            if not candidates:\n                candidates = self.unigram\n            if not candidates:\n                break\n            nxt = candidates.most_common(1)[0][0]\n            if nxt in [".", "!", "?"] and (not output):\n                break\n            output.append(nxt)\n            working.append(nxt)\n            if nxt in [".", "!", "?"]:\n                break\n        return detokenize(output)\n\n\ndef detokenize(token_list):\n    text = " ".join(token_list)\n    text = re.sub(r"\\s+([.,!?;])", r"\\1", text)\n    return text.strip()\n\n\nclass SuggestionEngine:\n    def __init__(self, history_df: pd.DataFrame):\n        self.df = history_df.copy()\n        self.df["user_input"] = self.df["user_input"].fillna("").map(normalize_text)\n        self.df["context"] = self.df["context"].fillna("general").str.lower()\n        self.corpus = self.df["user_input"].tolist()\n        self.ngram = NGramPredictor(n=3).fit(self.corpus)\n        self.vectorizer = None\n        self.matrix = None\n        if len([c for c in self.corpus if c]) >= 2:\n            self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")\n            self.matrix = self.vectorizer.fit_transform(self.corpus)\n\n    def similar_text(self, prompt: str, context: str):\n        if self.vectorizer is None or not prompt.strip():\n            return None\n        q = self.vectorizer.transform([prompt])\n        scores = cosine_similarity(q, self.matrix).ravel()\n        # Slightly favor the selected context.\n        context_mask = (self.df["context"] == context.lower()).to_numpy(dtype=float)\n        scores = scores + 0.05 * context_mask\n        idx = int(np.argmax(scores))\n        if scores[idx] <= 0.02:\n            return None\n        return self.corpus[idx]\n\n    def insight_summary(self):\n        all_tokens = [t for doc in self.corpus for t in tokens(doc) if len(t) > 3]\n        common = Counter(all_tokens).most_common(10)\n        context_counts = self.df["context"].value_counts().to_dict()\n        return common, context_counts\n\n    def suggestions(self, prompt: str, context: str, top_k: int = 5):\n        prompt = normalize_text(prompt)\n        context = (context or "general").lower()\n        candidates = []\n\n        continuation = self.ngram.predict_continuation(prompt, max_words=10)\n        if continuation:\n            candidates.append({"source": "ml_ngram", "text": continuation, "reason": "learned from repeated phrase patterns"})\n\n        similar = self.similar_text(prompt, context)\n        if similar:\n            # Use a concise phrase from the closest prior text instead of copying a full paragraph.\n            words = similar.split()\n            snippet = " ".join(words[: min(18, len(words))])\n            candidates.append({"source": "similarity", "text": snippet, "reason": "matched a similar past writing sample"})\n\n        for template in TEMPLATES.get(context, TEMPLATES.get("note", [])):\n            candidates.append({"source": "template", "text": template, "reason": f"common {context} pattern"})\n\n        # Remove duplicates while preserving order.\n        seen = set()\n        unique = []\n        for item in candidates:\n            key = item["text"].lower().strip()\n            if key and key not in seen and key not in prompt.lower():\n                unique.append(item)\n                seen.add(key)\n        return unique[:top_k]\n\n\nclass BanditRanker:\n    """Simple reinforcement-learning style ranker.\n\n    Each suggestion source is an arm. Accepting a suggestion increases reward;\n    rejecting it decreases confidence. This is a lightweight demonstration of\n    feedback optimization, not a full RL agent.\n    """\n\n    def __init__(self, state):\n        self.state = state\n        self.state.setdefault("bandit", {})\n\n    def score(self, source):\n        arm = self.state["bandit"].setdefault(source, {"accepted": 1, "shown": 2})\n        return arm["accepted"] / max(arm["shown"], 1)\n\n    def rank(self, suggestions):\n        return sorted(suggestions, key=lambda s: self.score(s["source"]), reverse=True)\n\n    def update(self, source, accepted: bool):\n        arm = self.state["bandit"].setdefault(source, {"accepted": 1, "shown": 2})\n        arm["shown"] += 1\n        if accepted:\n            arm["accepted"] += 1\n\n\ndef load_history(uploaded_file):\n    if uploaded_file is not None:\n        df = pd.read_csv(uploaded_file)\n        if "user_input" not in df.columns:\n            st.error("CSV must include a \'user_input\' column. Optional column: \'context\'.")\n            return pd.DataFrame(DEFAULT_EXAMPLES, columns=["context", "user_input"])\n        if "context" not in df.columns:\n            df["context"] = "general"\n        return df[["context", "user_input"]]\n    return pd.DataFrame(DEFAULT_EXAMPLES, columns=["context", "user_input"])\n\n\nst.set_page_config(page_title=APP_NAME, page_icon="✍️", layout="wide")\nst.title(APP_NAME)\nst.markdown(f"**Author:** {AUTHOR_NAME} &nbsp;&nbsp; | &nbsp;&nbsp; **Mentor:** {MENTOR_NAME}")\nst.caption("Portfolio prototype: personalized next phrase suggestions for writing, email, notes, and search.")\n\nwith st.sidebar:\n    st.header(APP_NAME)\n    st.markdown(f"**Author:** {AUTHOR_NAME}")\n    st.markdown(f"**Mentor:** {MENTOR_NAME}")\n    st.divider()\n    st.header("Privacy-first prototype")\n    st.write("This demo keeps examples in the current session unless you upload or download data yourself.")\n    uploaded = st.file_uploader("Upload optional writing history CSV", type=["csv"])\n    st.write("Required CSV column: `user_input`; optional: `context`.")\n\nhistory = load_history(uploaded)\nif "session_history" not in st.session_state:\n    st.session_state.session_history = []\nif st.session_state.session_history:\n    history = pd.concat([history, pd.DataFrame(st.session_state.session_history)], ignore_index=True)\n\nengine = SuggestionEngine(history)\nranker = BanditRanker(st.session_state)\n\nleft, right = st.columns([2, 1])\nwith left:\n    context = st.selectbox("Writing context", ["email", "search", "note", "report", "general"], index=0)\n    prompt = st.text_area("Start typing", height=160, placeholder="Type a sentence, email, note, or search query...")\n    add_to_history = st.checkbox("Add this text to my session history after generating suggestions", value=True)\n    if st.button("Generate suggestions", type="primary"):\n        if add_to_history and prompt.strip():\n            st.session_state.session_history.append({"context": context, "user_input": prompt.strip(), "timestamp": datetime.utcnow().isoformat()})\n        st.session_state.last_suggestions = ranker.rank(engine.suggestions(prompt, context, top_k=5))\n\n    suggestions = st.session_state.get("last_suggestions", [])\n    if suggestions:\n        st.subheader("Suggested continuations")\n        for i, item in enumerate(suggestions, start=1):\n            st.markdown(f"**{i}.** {item[\'text\']}")\n            st.caption(f"Source: {item[\'source\']} — {item[\'reason\']}")\n            c1, c2 = st.columns([1, 1])\n            with c1:\n                if st.button(f"Accept #{i}", key=f"accept_{i}"):\n                    ranker.update(item["source"], True)\n                    st.success("Feedback saved. This source will rank higher next time.")\n            with c2:\n                if st.button(f"Reject #{i}", key=f"reject_{i}"):\n                    ranker.update(item["source"], False)\n                    st.info("Feedback saved. This source will rank lower next time.")\n\nwith right:\n    st.subheader("Behavior insights")\n    common, counts = engine.insight_summary()\n    st.write("Contexts learned:")\n    st.json(counts)\n    st.write("Common terms:")\n    st.write(", ".join([w for w, _ in common]) if common else "No terms yet.")\n    st.subheader("Feedback model")\n    st.json(st.session_state.get("bandit", {}))\n\nst.divider()\nprofile = {\n    "created_at": datetime.utcnow().isoformat(),\n    "session_history": st.session_state.get("session_history", []),\n    "bandit": st.session_state.get("bandit", {}),\n}\nst.download_button(\n    "Download my session profile JSON",\n    data=json.dumps(profile, indent=2),\n    file_name="genai_prompt_assistant_profile.json",\n    mime="application/json",\n)\nst.caption("Prototype limitation: this demo is not a keyboard extension. iOS keyboard integration requires a native Swift/SwiftUI app and explicit user privacy controls.")\n'
(PROJECT_DIR / "app" / "app.py").write_text(app_code, encoding="utf-8")
print("Wrote", PROJECT_DIR / "app" / "app.py")

Wrote yashvi_ai_prompt_assistant/app/app.py


# 11. Run Streamlit from Colab

After writing `app.py`, you can run it in Colab. The `localtunnel` command prints a public demo URL. For a real portfolio, deploy from GitHub to Streamlit Community Cloud or another hosting service.

In [ ]:
# Run these commands in Colab when you want a temporary live demo.
!pip -q install streamlit
!npm install -g localtunnel
!streamlit run genai_prompt_assistant/app/app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 11.18.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.18.0
npm notice To update run: npm install -g npm@11.18.0
npm notice
⠸⠙⠹⠸⠼⠴⠦⠧

your url is: https://cool-points-lead.loca.lt
2026-07-07 21:53:33.629 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.222.165.148:8501

